In [ ]:
import os
import shutil
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from matplotlib.ticker import MaxNLocator

plt.style.use('seaborn-v0_8-paper')

In [ ]:
def collect_convergence(network_names_e, network_names_g, 
                        slack_positions_g, slack_position_e,
                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                        p2g_units_max, gfg_units_max,
                        lin_solver, lin_solver_parameters,
                        disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
    
    for network_name_e in network_names_e:            
        for network_name_g in network_names_g:
            for slack_position_g in slack_positions_g:
                header = ""
                rows = {}

                for number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list):
                    header += " & ({:d}, {:d}) ({:d}, {:d})".format(number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g)
                    for p2g_units in range(0, p2g_units_max+1):
                        for gfg_units in range(0, gfg_units_max+1):
                                         
                            if lin_solver in {'direct', 'lu'}:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver)
                            else:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver,
                                                         preconditioner_tag)
                            
                            if rows.get((p2g_units, gfg_units)) is None:
                                rows[(p2g_units, gfg_units)] = ""
                            
                            try:
                                summary = np.genfromtxt(os.path.join(directory, 'summary.txt'),  dtype=str, delimiter='\n')
                                iterations = float(summary[1].split()[-1])
                                residual = float(summary[2].split()[-1])
                                coupling_units_okay = float(summary[-3].split()[-1])                                   
                                                            
                                if (residual < 1e-6) and (coupling_units_okay == 1):
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{green!50}} {:3d}".format(int(iterations))
                                elif (residual < 1e-6) and (coupling_units_okay != 1):
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{orange!50}} {:3d}".format(int(iterations))
                                else:
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{red!50}} {:3d}".format(int(iterations))
                            except:
                                rows[(p2g_units, gfg_units)] += " & "

                header = "P2G & GFG" + header
                header += " \\\\ \\hline"
                print(header)

                for key, value in rows.items():
                    rows[key] += " \\\\ \\hline"
                    rows[key] = "{:d} & {:d}".format(key[0], key[1]) + rows[key]
                    print(rows[key])


def collect_convergence_fixed_size(network_names_e, network_names_g, 
                                   slack_positions_g, slack_position_e,
                                   number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c_list,
                                   p2g_units_max, gfg_units_max,
                                   lin_solver, lin_solver_parameters,
                                   disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
    
    for network_name_e in network_names_e:            
        for network_name_g in network_names_g:
            for slack_position_g in slack_positions_g:
                for number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list):
                    rows = {}
                    header = ""
                    for p2g_units in range(0, p2g_units_max+1):
                        for gfg_units in range(0, gfg_units_max+1):
                            stop_clones_c = 0
                            for number_of_clones_c in number_of_clones_c_list:
                                if lin_solver in {'direct', 'lu'}:
                                    directory = os.path.join(disk, 
                                                             results,
                                                             '{}_{}'.format(network_name_e, network_name_g),
                                                             'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                             'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                                  number_of_clones_g,
                                                                                                                                  number_of_clones_c, 
                                                                                                                                  number_of_merges_e, 
                                                                                                                                  number_of_merges_g),
                                                             'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                             lin_solver)
                                else:
                                    directory = os.path.join(disk, 
                                                             results,
                                                             '{}_{}'.format(network_name_e, network_name_g),
                                                             'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                             'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                                  number_of_clones_g,
                                                                                                                                  number_of_clones_c, 
                                                                                                                                  number_of_merges_e, 
                                                                                                                                  number_of_merges_g),
                                                             'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                             lin_solver,
                                                             preconditioner_tag)

                                if rows.get((p2g_units, gfg_units)) is None:
                                    rows[(p2g_units, gfg_units)] = ""

                                try:
                                    summary = np.genfromtxt(os.path.join(directory, 'summary.txt'),  dtype=str, delimiter='\n')
                                    iterations = float(summary[1].split()[-1])
                                    residual = float(summary[2].split()[-1])
                                    coupling_units_okay = float(summary[-3].split()[-1])

                                    stop_clones_c += 1

                                    if (residual < 1e-6) and (coupling_units_okay == 1):
                                        rows[(p2g_units, gfg_units)] += " & \\cellcolor{{green!50}} {:3d}".format(int(iterations))
                                    elif (residual < 1e-6) and (coupling_units_okay != 1):
                                        rows[(p2g_units, gfg_units)] += " & \\cellcolor{{orange!50}} {:3d}".format(int(iterations))
                                    else:
                                        rows[(p2g_units, gfg_units)] += " & \\cellcolor{{red!50}} {:3d}".format(int(iterations))
                                except:
                                    rows[(p2g_units, gfg_units)] += ""

                    
                    print("({:d}, {:d}) ({:d}, {:d})".format(number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g))
                    for number_of_clones_c in number_of_clones_c_list[:stop_clones_c]:
                        header += " & \\textbf{{{:d}}}".format(number_of_clones_c)
                    header = "P2G & GFG" + header
                    header += " \\\\ \\hline"
                    print(header)

                    for key, value in rows.items():
                        rows[key] += " \\\\ \\hline"
                        rows[key] = "{:d} & {:d}".format(key[0], key[1]) + rows[key]
                        print(rows[key])


def collect_difference(network_names_e, network_names_g, 
                       slack_positions_g, slack_position_e,
                       number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list,
                       p2g_units_max, gfg_units_max,
                       lin_solver, lin_solver_parameters,
                       disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
    
    for network_name_e in network_names_e:            
        for network_name_g in network_names_g:
            for slack_position_g in slack_positions_g:
                header = ""
                rows_difference = {}
                
                for number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list):
                    number_of_clones_c = max(number_of_clones_e, number_of_clones_g)
                    header += " & ({:d}, {:d}) ({:d}, {:d})".format(number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g)
                    for p2g_units in range(0, p2g_units_max+1):
                        for gfg_units in range(0, gfg_units_max+1):
                            if lin_solver in {'direct', 'lu'}:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver)
                            else:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver,
                                                         preconditioner_tag)
                                
                            if lin_solver in {'direct', 'lu'}:
                                directory_no_copies = os.path.join(disk, 
                                                                   results,
                                                                   '{}_{}'.format(network_name_e, network_name_g),
                                                                   'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                                   'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(1, 
                                                                                                                                        1,
                                                                                                                                        1, 
                                                                                                                                        0, 
                                                                                                                                        0),
                                                                   'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                                   lin_solver)
                            else:
                                directory_no_copies = os.path.join(disk, 
                                                                   results,
                                                                   '{}_{}'.format(network_name_e, network_name_g),
                                                                   'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                                   'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(1, 
                                                                                                                                        1,
                                                                                                                                        1, 
                                                                                                                                        0, 
                                                                                                                                        0),
                                                                   'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                                   lin_solver,
                                                                   preconditioner_tag)
                            
                            if rows_difference.get((p2g_units, gfg_units)) is None:
                                rows_difference[(p2g_units, gfg_units)] = ""
                            
                            try:
                                if (p2g_units == 0) and (gfg_units == 0):
                                    rows_difference[(p2g_units, gfg_units)]  = " & "
                                elif (number_of_clones_e == 1) or (number_of_clones_g == 1):
                                    rows_difference[((p2g_units, gfg_units))] = " & " 
                                else:
                                    summary = np.genfromtxt(os.path.join(directory, 'summary.txt'),  dtype=str, delimiter='\n')
                                    residual = float(summary[2].split()[-1])

                                    P_c = np.genfromtxt(os.path.join(directory, 'P_c.txt'),  dtype=str, delimiter='\n')
                                    for i in range(P_c.shape[0]):
                                        P_c[i] = P_c[i].split(' ')[-1]
                                    P_c = P_c.astype(float)

                                    P_c_no_copies = np.genfromtxt(os.path.join(directory_no_copies, 'P_c.txt'),  dtype=str, delimiter='\n')
                                    if P_c_no_copies.shape == ():
                                        P_c_no_copies = np.array([P_c_no_copies])
                                    for i in range(P_c_no_copies.shape[0]):
                                        P_c_no_copies[i] = P_c_no_copies[i].split(' ')[-1]
                                    P_c_no_copies = P_c_no_copies.astype(float)
                                    
                                    max_difference_per_copy = []
                                    if p2g_units > 0:
                                        for i in range(1, number_of_clones_e):
                                            # max_difference_per_copy.append(np.max(np.abs((P_c[:p2g_units]-P_c[i*p2g_units:(i+1)*p2g_units]) / P_c[:p2g_units])))
                                            max_difference_per_copy.append(np.max(np.abs((P_c_no_copies[:p2g_units] - P_c[i*p2g_units:(i+1)*p2g_units]) / P_c_no_copies[:p2g_units])))

                                    if gfg_units > 0:
                                        start = p2g_units * number_of_clones_e
                                        for i in range(1, number_of_clones_e):
                                            # max_difference_per_copy.append(np.max(np.abs((P_c[start:start+gfg_units]-P_c[start+i*gfg_units:start+(i+1)*gfg_units]) / P_c[start:start+gfg_units])))
                                            max_difference_per_copy.append(np.max(np.abs((P_c_no_copies[p2g_units:]-P_c[start+i*gfg_units:start+(i+1)*gfg_units]) / P_c_no_copies[p2g_units:])))
                                    max_difference = np.max(max_difference_per_copy)
                                    
                                    
                                    if residual <= 1e-6:
                                        rows_difference[(p2g_units, gfg_units)]  += " & {:8.3e}".format(max_difference)
                                    else:
                                        rows_difference[(p2g_units, gfg_units)]  += " & "
                            except Exception as err:
                                print(err)
                                rows_difference[(p2g_units, gfg_units)] += " & "

                header = "P2G & GFG" + header
                header += " \\\\ \\hline"
                print(header)

                for key, value in rows_difference.items():
                    rows_difference[key] += " \\\\ \\hline"
                    rows_difference[key] = "{:d} & {:d}".format(key[0], key[1]) + rows_difference[key]
                    print(rows_difference[key])


def collect_computational_time(network_names_e, network_names_g, 
                               slack_positions_g, slack_position_e,
                               number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                               p2g_units_max, gfg_units_max,
                               lin_solver, lin_solver_parameters,
                               disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
        
    data = {}
    data_average = {}
    data_unknowns = {}
    
    header = ""
    rows = {}
    for network_name_e in network_names_e:            
        for network_name_g in network_names_g:
            for slack_position_g in slack_positions_g:  
                for i, (number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g) in enumerate(zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)):
                    for p2g_units in range(0, p2g_units_max+1):
                        for gfg_units in range(0, gfg_units_max+1):         
                            if lin_solver in {'direct', 'lu'}:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver)
                            else:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver,
                                                         preconditioner_tag)
                                
                            if rows.get((p2g_units, gfg_units)) is None:
                                rows[(p2g_units, gfg_units)] = ""
                                        
                            try:
                                summary = np.genfromtxt(os.path.join(directory, 'summary.txt'),  dtype=str, delimiter='\n')
                                residual = float(summary[2].split()[-1])
                                coupling_units_okay = float(summary[-3].split()[-1])
                                average_linear_time = float(summary[6].split()[-1])
                                total_time = float(summary[-5].split()[-1])
                                unknowns =  float(summary[0].split()[-1])
                                                          
                                data[(p2g_units, gfg_units, i)] = total_time
                                data_average[(p2g_units, gfg_units, i)] = average_linear_time
                                data_unknowns[(p2g_units, gfg_units, i)] = int(unknowns)

                                if (residual < 1e-6) and (coupling_units_okay == 1):
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{green!50}} {:4.3e}".format(float(total_time))
                                elif (residual < 1e-6) and (coupling_units_okay != 1):
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{orange!50}} {:4.3e}".format(float(total_time))
                                else:
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{red!50}} {:4.3e}".format(float(total_time))
                            except:
                                rows[(p2g_units, gfg_units)] += " & "

                header = "P2G & GFG" + header
                header += " \\\\ \\hline"
                print(header)
                
                for key, value in rows.items():
                    rows[key] += " \\\\ \\hline"
                    rows[key] = "{:d} & {:d}".format(key[0], key[1]) + rows[key]
                    print(rows[key])
                            
    return data, data_average, data_unknowns


def find_order(x0=[], x1=[], option='lowest'):
    x = np.concatenate([x0, x1])
    
    if x.shape[0] > 0:
        if option in {'lowest'}:
            result = int(np.min(np.floor(np.log10(x))))
        else:
            result = int(np.max(np.ceil(np.log10(x))))
    else:
        result = 0

    return result


def plot_computational_time(network_names_e, network_names_g, 
                            slack_positions_g, slack_position_e,
                            number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                            p2g_units_max, gfg_units_max,
                            lin_solver, lin_solver_parameters,
                            disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
    
    data, data_average, data_unknowns = collect_computational_time(network_names_e, network_names_g, 
                                                                   slack_positions_g, slack_position_e,
                                                                   number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                                   p2g_units_max, gfg_units_max,
                                                                   lin_solver, lin_solver_parameters,
                                                                   disk, results)
    
    sizes = set([key[-1] for key in data.keys()])
    
    fontsize = 11
    
    for p2g_units in range(p2g_units_max+1):
        for gfg_units in range(gfg_units_max+1):
            values = []
            values_average = []
            unknowns = []
            for size in sizes:
                if data.get((p2g_units, gfg_units, size)) is not None:
                    if data[(p2g_units, gfg_units, size)] > 0:
                        values.append(data[(p2g_units, gfg_units, size)])
                        values_average.append(data_average[(p2g_units, gfg_units, size)])
                        unknowns.append(data_unknowns[(p2g_units, gfg_units, size)])
                
            fig, ax = plt.subplots(1, 2, figsize=(12, 5))

            # total computational time
            ax[0].loglog(unknowns, values, marker='.', label="{}".format(lin_solver))

            lowest_order = find_order(x0=unknowns, option='lowest') # int(np.floor(np.log10(unknowns[0])))
            highest_order = find_order(x0=unknowns, option='highest') # int(np.ceil(np.log10(unknowns[-1])))
            
            if lin_solver in {'direct', 'lu'}:
                ax[0].loglog([10**k for k in range(lowest_order, highest_order+1)], [10**(k-3.5) for k in range(lowest_order, highest_order+1)], 
                                linestyle='--', label="$O(n)$")
            else:
                ax[0].loglog([10**k for k in range(lowest_order, highest_order+1)], [10**(2*k-7.5) for k in range(lowest_order, highest_order+1)], 
                                linestyle='--', label="$O(n^2)$")

            ax[0].set_xticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

            lowest_order = find_order(x0=values, option='lowest') # int(np.floor(np.log10(values[0])))
            highest_order = find_order(x0=values, option='highest') # int(np.ceil(np.log10(values[-1])))
            ax[0].set_yticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

            ax[0].set_title("Total computational time of NR", fontsize=fontsize)
            ax[0].set_xlabel("Number of unknowns", fontsize=fontsize)
            ax[0].set_ylabel("Time in seconds", fontsize=fontsize)
            
            ax[0].tick_params(axis='both', which='major', labelsize=fontsize)
            ax[0].tick_params(axis='both', which='minor', labelsize=fontsize)

            ax[0].grid()

            ax[0].legend(loc='upper left', fontsize=fontsize)

            # average linear time
            ax[1].loglog(unknowns, values_average, marker='.', label="{}".format(lin_solver))

            lowest_order = find_order(x0=unknowns, option='lowest') # int(np.floor(np.log10(unknowns[0])))
            highest_order = find_order(x0=unknowns, option='highest') # int(np.ceil(np.log10(unknowns[-1])))
            
            if lin_solver in {'direct', 'lu'}:
                ax[1].loglog([10**k for k in range(lowest_order, highest_order+1)], [10**(k-5) for k in range(lowest_order, highest_order+1)], 
                                linestyle='--', label="$O(n)$")
            else:
                ax[1].loglog([10**k for k in range(lowest_order, highest_order+1)], [10**(2*k-9) for k in range(lowest_order, highest_order+1)], 
                                linestyle='--', label="$O(n^2)$")

            ax[1].set_xticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

            lowest_order = find_order(x0=values_average, option='lowest') # int(np.floor(np.log10(values[0])))
            highest_order = find_order(x0=values_average, option='highest') # int(np.ceil(np.log10(values[-1])))
            ax[1].set_yticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{{{}}}$".format(k) for k in range(lowest_order, highest_order+1)])

            ax[1].set_title("Average time of linear solver", fontsize=fontsize)
            ax[1].set_xlabel("Number of unknowns", fontsize=fontsize)
            ax[1].set_ylabel("Time in seconds", fontsize=fontsize)
            
            ax[1].tick_params(axis='both', which='major', labelsize=fontsize)
            ax[1].tick_params(axis='both', which='minor', labelsize=fontsize)


            ax[1].grid()

            ax[1].legend(loc='upper left', fontsize=fontsize)

            plt.suptitle("P2G = {}, GFG = {}".format(p2g_units, gfg_units), fontsize=fontsize+3)
            
            directory = os.path.join(disk, 
                                     results,
                                     '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                     'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                     lin_solver)
            
            if lin_solver not in {'direct', 'lu'}:
                directory = os.path.join(directory, preconditioner_tag)
                
            directory = os.path.join(directory, 'computational_time')
            
            os.makedirs(directory, exist_ok=True)

            name = "total_computational_time_NR_and_average_linear_time_clones_c_{}_p2g_{}_gfg_{}".format(number_of_clones_c, p2g_units, gfg_units)
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
                            
                
def collect_residuals(network_names_e, network_names_g,
                      slack_positions_g, slack_position_e,
                      number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                      p2g_units_max, gfg_units_max,
                      lin_solver, lin_solver_parameters,
                      disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
 
    header = ""
    rows = {}
    rows_difference = {}

    data = {}
    data_unknowns = {}

    for i, (number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g) in enumerate(zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)):
        header += " & ({:d}, {:d}) ({:d}, {:d})".format(number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g)
        for network_name_e in network_names_e:            
            for network_name_g in network_names_g:
                for p2g_units in range(0, p2g_units_max+1):
                    for gfg_units in range(0, gfg_units_max+1):
                        for slack_position_g in slack_positions_g:                    
                            if lin_solver in {'direct', 'lu'}:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver)
                            else:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver,
                                                         preconditioner_tag)
                            
                            if rows.get((p2g_units, gfg_units)) is None:
                                rows[(p2g_units, gfg_units)] = ""
                            if rows_difference.get((p2g_units, gfg_units)) is None:
                                rows_difference[(p2g_units, gfg_units)] = ""
                            
                            try:
                                summary = np.genfromtxt(os.path.join(directory, 'summary.txt'),  dtype=str, delimiter='\n')
                                residual = float(summary[2].split()[-1])
                                coupling_units_okay = float(summary[-3].split()[-1])
                                unknowns =  float(summary[0].split()[-1])
                                
                                data[(p2g_units, gfg_units, i)] = np.loadtxt(os.path.join(directory, 'residuals.txt'))
                                data_unknowns[(p2g_units, gfg_units, i)] = int(unknowns)
                                                                                                                    
                                if (residual < 1e-6) and (coupling_units_okay == 1):
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{green!50}} {:4.3e}".format(float(residual))
                                elif (residual < 1e-6) and (coupling_units_okay != 1):
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{orange!50}} {:4.3e}".format(float(residual))
                                else:
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{red!50}} {:4.3e}".format(float(residual))
                            except:
                                rows[(p2g_units, gfg_units)] += " & "
                                rows_difference[(p2g_units, gfg_units)] += " & "

    header = "P2G & GFG" + header
    header += " \\\\ \\hline"
    print(header)

    for key, value in rows.items():
        rows[key] += " \\\\ \\hline"
        rows[key] = "{:d} & {:d}".format(key[0], key[1]) + rows[key]
        print(rows[key])
        
    return data, data_unknowns
 

def plot_residuals_fixed_size(network_names_e, network_names_g,
                              slack_positions_g, slack_position_e,
                              number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                              p2g_units_max, gfg_units_max,
                              lin_solver, lin_solver_parameters,
                              disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
    
    data, data_unknowns = collect_residuals(network_names_e, network_names_g,
                                            slack_positions_g, slack_position_e,
                                            number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                            p2g_units_max, gfg_units_max,
                                            lin_solver, lin_solver_parameters,
                                            disk, results)
    
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    linestyles = ['solid', 'dashed', 'solid', 'dashed']
    markers = ['', '', '^', '^']
    fontsize = 12
    
    for i in range(len(number_of_clones_e_list)):
        present = False
        
        ax = plt.figure(figsize=(9, 9)).gca()
                
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        
        x_max = 1
        
        for p2g_units in range(0, p2g_units_max+1):
            for gfg_units in range(0, gfg_units_max+1):
                try:      
                    if data[(p2g_units, gfg_units, i)][-1] < 10**-1:
                        plt.semilogy(data[(p2g_units, gfg_units, i)],
                                     linestyle=linestyles[p2g_units], marker=markers[p2g_units],
                                     label="P2G = {}, GFG = {}".format(p2g_units, gfg_units))
                        
                        x_max = len(data[(p2g_units, gfg_units, i)]) if x_max < len(data[(p2g_units, gfg_units, i)]) else x_max
                        
                        present = True
                except:
                    pass
        
        if present:            
            plt.hlines(10**-6, plt.xlim()[0], plt.xlim()[1], linestyles=':', colors='orange')
            
            plt.title("NR Residual of case {}".format(titles[i]), fontsize=fontsize+3)
            plt.xlabel("Iteration", fontsize=fontsize)
            plt.ylabel(r"$\|F\|_2$", fontsize=fontsize)
            
            plt.xlim(0, x_max)
            
            x_step = max(1, 10**int(np.log10(plt.xlim()[1])-1)) * (5 if int(str(int(plt.xlim()[1]))[:2]) > 20 else 1)
            plt.xticks(np.arange(plt.xlim()[0], plt.xlim()[1], x_step, dtype=int))
            plt.yticks([10**k for k in np.arange(int(np.log10(plt.ylim()[0])), int(np.log10(plt.ylim()[1]))+1, 1, dtype=float)])
            
            plt.tick_params(axis='both', which='major', labelsize=fontsize)
            plt.tick_params(axis='both', which='minor', labelsize=fontsize)
                
            plt.grid()
            plt.legend(loc='upper left', fontsize=fontsize-2, ncols=4, bbox_to_anchor=(-0.1, -0.1))
                        
            directory = os.path.join(disk, 
                                     results,
                                     '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                     'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                     lin_solver)
            
            if lin_solver not in {'direct', 'lu'}:
                directory = os.path.join(directory, preconditioner_tag)
                
            directory = os.path.join(directory, 'residuals')
            
            os.makedirs(directory, exist_ok=True)

            name = "residuals_NR_clones_e_{}_clones_g_{}_clones_c_{}_merges_e{}_merges_g_{}".format(number_of_clones_e_list[i],
                                                                                                    number_of_clones_g_list[i], 
                                                                                                    number_of_clones_c, 
                                                                                                    number_of_merges_e_list[i], 
                                                                                                    number_of_merges_g_list[i])
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
        else:
            plt.close()
        

def plot_residuals_fixed_coupling(network_names_e, network_names_g,
                                  slack_positions_g, slack_position_e,
                                  number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                  p2g_units_max, gfg_units_max,
                                  lin_solver, lin_solver_parameters,
                                  disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
    
    data, data_unknowns = collect_residuals(network_names_e, network_names_g,
                                            slack_positions_g, slack_position_e,
                                            number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                            p2g_units_max, gfg_units_max,
                                            lin_solver, lin_solver_parameters,
                                            disk, results)

    labels = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    linestyles = ['solid', 'dashed', 'solid', 'dashed', 'solid', 'dashed']
    markers = ['', '', '^', '^', 'o', 'o']
    fontsize = 12

    for p2g_units in range(p2g_units_max+1):
        for gfg_units in range(gfg_units_max+1):
            x_max = 1
            
            ax = plt.figure(figsize=(7, 7)).gca()
            
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))

            for i in range(len(number_of_clones_e_list)):
                try:
                    if data[(p2g_units, gfg_units, i)][-1] < 10**-1:
                        plt.semilogy(data[(p2g_units, gfg_units, i)],
                                     linestyle=linestyles[i], marker=markers[i],
                                     label="{}".format(labels[i]))
                        x_max = len(data[(p2g_units, gfg_units, i)]) if x_max < len(data[(p2g_units, gfg_units, i)]) else x_max
                except:
                    pass
            
            plt.hlines(10**-6, plt.xlim()[0], plt.xlim()[1], linestyles=':', colors='orange')
            
            plt.grid()
            
            plt.title("P2G = {}, GFG = {}".format(p2g_units, gfg_units), fontsize=fontsize+3)
            plt.xlabel("Iteration", fontsize=fontsize)
            plt.ylabel(r"$\|F\|_2$", fontsize=fontsize)
            
            plt.xlim(0, x_max)
            
            x_step = max(1, 10**int(np.log10(plt.xlim()[1])-1)) * (5 if int(str(int(plt.xlim()[1]))[:2]) > 20 else 1)
            plt.xticks(np.arange(plt.xlim()[0], plt.xlim()[1], x_step, dtype=int))
            plt.yticks([10**float(k) for k in np.arange(int(np.log10(plt.ylim()[0])), int(np.log10(plt.ylim()[1])+1), 1)])
            plt.yticks([10**k for k in np.arange(int(np.log10(plt.ylim()[0])), int(np.log10(plt.ylim()[1]))+1, 1, dtype=float)])
            
            plt.tick_params(axis='both', which='major', labelsize=fontsize)
            plt.tick_params(axis='both', which='minor', labelsize=fontsize)
            
            plt.legend(loc='upper right', fontsize=fontsize-2)
            
            directory = os.path.join(disk, 
                                     results,
                                     '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                     'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                     lin_solver)
            
            if lin_solver not in {'direct', 'lu'}:
                directory = os.path.join(directory, preconditioner_tag)
                
            directory = os.path.join(directory, 'residuals')
            
            os.makedirs(directory, exist_ok=True)

            name = "residuals_NR_clones_c_{}_p2g_{}_gfg_{}".format(number_of_clones_c, p2g_units, gfg_units)
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
            

def collect_residuals_krylov(network_names_e, network_names_g,
                             slack_positions_g, slack_position_e,
                             number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                             p2g_units_max, gfg_units_max,
                             lin_solver, lin_solver_parameters,
                             disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
    
    data = {}
 
    for i, (number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g) in enumerate(zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)):
        for network_name_e in network_names_e:            
            for network_name_g in network_names_g:
                for p2g_units in range(0, p2g_units_max+1):
                    for gfg_units in range(0, gfg_units_max+1):
                        for slack_position_g in slack_positions_g:                  
                            if lin_solver in {'direct', 'lu'}:
                                print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
                                return None
                            else:
                                data[(p2g_units, gfg_units, i)] = []
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver,
                                                         preconditioner_tag,
                                                         'residuals_krylov')
                                
                                try:
                                    files = os.listdir(directory)
                                    files.sort()
                                    for file in files:
                                        data[(p2g_units, gfg_units, i)].append(np.loadtxt(os.path.join(directory, '{}'.format(file))))
                                except:
                                    pass                                                                                 

    return data


def plot_preconditioned_residuals_krylov_per_case(network_names_e, network_names_g,
                                                  slack_positions_g, slack_position_e,
                                                  number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                  p2g_units_max, gfg_units_max,
                                                  lin_solver, lin_solver_parameters,
                                                  disk, results):
    if lin_solver in {'direct', 'lu'}:
        print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
        return None
    else:
        try:
            if lin_solver_parameters['preconditioner'] == 'ilu':
                preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
            else:
                preconditioner_tag = lin_solver_parameters['preconditioner']
        except:
            pass
        
        directory = os.path.join(disk, 
                                 results,
                                 '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                 'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                 lin_solver,
                                 preconditioner_tag,
                                 'residuals_krylov')
    
    data = collect_residuals_krylov(network_names_e, network_names_g,
                                    slack_positions_g, slack_position_e,
                                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                    p2g_units_max, gfg_units_max,
                                    lin_solver, lin_solver_parameters,
                                    disk, results)
    
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    fontsize = 12
    
    for key, value in data.items():
        if (len(value) > 0):
            x_max = 1
            
            cmap = mpl.colormaps['copper']
            colors = cmap(np.linspace(0, 1, len(value)))
            
            plt.figure(figsize=(7, 7))
            
            for i, row in enumerate(value):
                plt.plot(row[:, -1], linestyle='solid', marker='*', label='NR = {}'.format(i+1), color=colors[i])
                x_max = len(row[:, -1]) if x_max < len(row[:, -1]) else x_max

            plt.title("{} | P2G = {}, GFG = {}: preconditioned residual at each NR iteration".format(titles[key[2]], key[0], key[1]), fontsize=fontsize+1)
            plt.xlabel("GMRES Iterations", fontsize=fontsize)
            plt.ylabel("Residual", fontsize=fontsize)

            plt.yscale('log', base=10)
            
            plt.xlim(0, x_max)
            
            x_step = max(1, 10**int(np.log10(plt.xlim()[1])-1)) * (5 if int(str(int(plt.xlim()[1]))[:2]) > 20 else 1)
            plt.xticks(np.arange(plt.xlim()[0], plt.xlim()[1], x_step, dtype=int))
            plt.yticks([10**float(k) for k in np.arange(int(np.log10(plt.ylim()[0])), int(np.log10(plt.ylim()[1])+1), 1)])
            
            plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
            plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)
                        
            plt.grid()
            
            plt.legend(loc='upper left', bbox_to_anchor=(-0.05, -0.1), ncols=5, fontsize=fontsize-2)
            
            name = "preconditioned_residuals_NR_clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}_p2g_{}_gfg_{}".format(number_of_clones_e_list[key[2]],
                                                                                                                                  number_of_clones_g_list[key[2]],
                                                                                                                                  number_of_clones_c, 
                                                                                                                                  number_of_merges_e_list[key[2]],
                                                                                                                                  number_of_merges_g_list[key[2]],
                                                                                                                                  key[0], 
                                                                                                                                  key[1])
            os.makedirs(directory, exist_ok=True)
            
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")

    
def plot_preconditioned_residuals_krylov_at_all_NR_iterations(network_names_e, network_names_g,
                                                              slack_positions_g, slack_position_e,
                                                              number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                              p2g_units_max, gfg_units_max,
                                                              lin_solver, lin_solver_parameters,
                                                              disk, results):
    if lin_solver in {'direct', 'lu'}:
        print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
        return None
    else:
        try:
            if lin_solver_parameters['preconditioner'] == 'ilu':
                preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
            else:
                preconditioner_tag = lin_solver_parameters['preconditioner']
        except:
            pass
        
        directory = os.path.join(disk, 
                                 results,
                                 '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                 'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                 lin_solver,
                                 preconditioner_tag,
                                 'residuals_krylov')
        
    data = collect_residuals_krylov(network_names_e, network_names_g,
                                    slack_positions_g, slack_position_e,
                                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                    p2g_units_max, gfg_units_max,
                                    lin_solver, lin_solver_parameters,
                                    disk, results)
    
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    fontsize = 12

    for key, value in data.items():
        if (len(value) > 0):
            residuals = np.array([])
            iterations = [0]
            for row in value:
                iterations.append(len(row[:, -1]))
                residuals = np.concatenate([residuals, row[:, -1]])

            plt.figure(figsize=(12, 6.75))

            plt.semilogy(residuals, linestyle='solid', marker='*')
            plt.vlines(np.cumsum(iterations)[:-1], ymin=plt.ylim()[0], ymax=plt.ylim()[1],
                       linestyles='dashed', colors='orange')

            plt.title("P2G = {}, GFG = {}, {}: preconditioned residual at all NR iterations".format(key[0], key[1], titles[key[2]]), fontsize=fontsize+3)
            plt.xlabel("GMRES Iterations", fontsize=fontsize)
            plt.ylabel("Residual", fontsize=fontsize)
            
            plt.xlim(0, len(residuals))
            
            x_step = max(1, 10**int(np.log10(plt.xlim()[1])-1)) * (5 if int(str(int(plt.xlim()[1]))[:2]) > 20 else 1)
            plt.xticks(np.arange(plt.xlim()[0], plt.xlim()[1], x_step, dtype=int))
            plt.yticks([10**float(k) for k in np.arange(int(np.log10(plt.ylim()[0])), int(np.log10(plt.ylim()[1])+1), 1)])
            
            plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
            plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)
            
            plt.grid()
            
            name = "preconditioned_residuals_at_all_NR_iterations_clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}_p2g_{}_gfg_{}".format(number_of_clones_e_list[key[2]],
                                                                                                                                                    number_of_clones_g_list[key[2]],
                                                                                                                                                    number_of_clones_c, 
                                                                                                                                                    number_of_merges_e_list[key[2]],
                                                                                                                                                    number_of_merges_g_list[key[2]],
                                                                                                                                                    key[0], 
                                                                                                                                                    key[1])
            os.makedirs(directory, exist_ok=True)
            
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
            
    
def plot_absolute_residuals_krylov_per_case(network_names_e, network_names_g,
                                            slack_positions_g, slack_position_e,
                                            number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                            p2g_units_max, gfg_units_max,
                                            lin_solver, lin_solver_parameters,
                                            disk, results):
    if lin_solver in {'direct', 'lu'}:
        print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
        return None
    else:
        try:
            if lin_solver_parameters['preconditioner'] == 'ilu':
                preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
            else:
                preconditioner_tag = lin_solver_parameters['preconditioner']
        except:
            pass
        
        directory = os.path.join(disk, 
                                 results,
                                 '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                 'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                 lin_solver,
                                 preconditioner_tag,
                                 'residuals_krylov')
        
    data = collect_residuals_krylov(network_names_e, network_names_g,
                                    slack_positions_g, slack_position_e,
                                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                    p2g_units_max, gfg_units_max,
                                    lin_solver, lin_solver_parameters,
                                    disk, results)
    
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    fontsize = 12
    
    for key, value in data.items():
        if (len(value) > 0):
            plt.figure(figsize=(7, 7))
            
            x_max = 0
            
            cmap = mpl.colormaps['copper']
            colors = cmap(np.linspace(0, 1, len(value)))
            
            for i, row in enumerate(value):
                residuals = []
                last_residual = 0
                
                for residual in row[:, -2]:
                    if residual > 0:
                        residuals.append(residual)
                        last_residual = residuals[-1]
                    else:
                        residuals.append(last_residual)
                        
                plt.semilogy(residuals, linestyle='solid', marker='*', label='NR = {}'.format(i+1), color=colors[i])
                
                x_max = len(residuals) if x_max < len(residuals) else x_max

            plt.title("P2G = {}, GFG = {}, {}: residual at each NR iteration".format(key[0], key[1], titles[key[2]]), fontsize=fontsize+3)
            plt.xlabel("GMRES Iterations", fontsize=fontsize)
            plt.ylabel("Residual", fontsize=fontsize)
            
            plt.xlim(0, x_max)
            
            x_step = max(1, 10**int(np.log10(plt.xlim()[1])-1)) * (5 if int(str(int(plt.xlim()[1]))[:2]) > 20 else 1)
            plt.xticks(np.arange(plt.xlim()[0], plt.xlim()[1], x_step, dtype=int))
            plt.yticks([10**float(k) for k in np.arange(int(np.log10(plt.ylim()[0])), int(np.log10(plt.ylim()[1])+1), 1)])
            
            plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
            plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)
            
            plt.grid()
            
            # plt.legend(loc='upper right', bbox_to_anchor=(1.125, 1), fontsize=fontsize-2)
            plt.legend(loc='upper left', fontsize=fontsize-2, ncols=5, bbox_to_anchor=(-0.075, -0.1))
            
            name = "residuals_clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}_p2g_{}_gfg_{}".format(number_of_clones_e_list[key[2]],
                                                                                                                number_of_clones_g_list[key[2]],
                                                                                                                number_of_clones_c, 
                                                                                                                number_of_merges_e_list[key[2]],
                                                                                                                number_of_merges_g_list[key[2]],
                                                                                                                key[0], 
                                                                                                                key[1])
            os.makedirs(directory, exist_ok=True)
            
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
    
            
def plot_absolute_residuals_krylov_at_all_NR_iterations(network_names_e, network_names_g,
                                                        slack_positions_g, slack_position_e,
                                                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                        p2g_units_max, gfg_units_max,
                                                        lin_solver, lin_solver_parameters,
                                                        disk, results):
    if lin_solver in {'direct', 'lu'}:
        print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
        return None
    else:
        try:
            if lin_solver_parameters['preconditioner'] == 'ilu':
                preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
            else:
                preconditioner_tag = lin_solver_parameters['preconditioner']
        except:
            pass
        
        directory = os.path.join(disk, 
                                 results,
                                 '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                 'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                 lin_solver,
                                 preconditioner_tag,
                                 'residuals_krylov')
        
    data = collect_residuals_krylov(network_names_e, network_names_g,
                                    slack_positions_g, slack_position_e,
                                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                    p2g_units_max, gfg_units_max,
                                    lin_solver, lin_solver_parameters,
                                    disk, results)
    
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    fontsize = 12
    
    for key, value in data.items():
        if (len(value) > 0):
            residuals = []
            iterations = [0]
            last_residual = 0
            for row in value:
                iterations.append(len(row[:, -1]))
                for residual in row[:, -2]:
                    if residual > 0:
                        residuals.append(residual)
                        last_residual = residuals[-1]
                    else:
                        residuals.append(last_residual)
                        
            plt.figure(figsize=(12, 6.75))

            plt.semilogy(residuals, linestyle='solid', marker='*')
            plt.vlines(np.cumsum(iterations)[:-1], ymin=plt.ylim()[0], ymax=plt.ylim()[1],
                       linestyles='dashed', colors='orange')

            plt.title("P2G = {}, GFG = {}, {}: residual at each NR iteration".format(key[0], key[1], titles[key[2]]), fontsize=fontsize+3)
            plt.xlabel("GMRES Iterations", fontsize=fontsize)
            plt.ylabel("Residual", fontsize=fontsize)
            
            plt.xlim(0, len(residuals))
            
            x_step = max(1, 10**int(np.log10(plt.xlim()[1])-1)) * (5 if int(str(int(plt.xlim()[1]))[:2]) > 20 else 1)
            plt.xticks(np.arange(plt.xlim()[0], plt.xlim()[1], x_step, dtype=int))
            plt.yticks([10**float(k) for k in np.arange(int(np.log10(plt.ylim()[0])), int(np.log10(plt.ylim()[1])+1), 1)])
            
            plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
            plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)
            
            plt.grid()
            
            name = "residuals_at_all_NR_iterations_clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}_p2g_{}_gfg_{}".format(number_of_clones_e_list[key[2]],
                                                                                                                                     number_of_clones_g_list[key[2]],
                                                                                                                                     number_of_clones_c, 
                                                                                                                                     number_of_merges_e_list[key[2]],
                                                                                                                                     number_of_merges_g_list[key[2]],
                                                                                                                                     key[0], 
                                                                                                                                     key[1])
            os.makedirs(directory, exist_ok=True)
            
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
            

def plot_iterations_krylov_per_case(network_names_e, network_names_g,
                                    slack_positions_g, slack_position_e,
                                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                    p2g_units_max, gfg_units_max,
                                    lin_solver, lin_solver_parameters,
                                    disk, results):
    if lin_solver in {'direct', 'lu'}:
        print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
        return None
    else:
        try:
            if lin_solver_parameters['preconditioner'] == 'ilu':
                preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
            else:
                preconditioner_tag = lin_solver_parameters['preconditioner']
        except:
            pass
        
        directory = os.path.join(disk, 
                                 results,
                                 '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                 'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                 lin_solver,
                                 preconditioner_tag,
                                 'iterations_krylov')
        
    data = collect_residuals_krylov(network_names_e, network_names_g,
                                    slack_positions_g, slack_position_e,
                                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                    p2g_units_max, gfg_units_max,
                                    lin_solver, lin_solver_parameters,
                                    disk, results)
    
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    fontsize = 12

    for key, value in data.items():
        if (len(value) > 0):
            iterations = [0]
            for row in value:
                iterations.append(len(row[:, -1])-1)

            plt.figure(figsize=(7, 7))
            
            plt.step(np.arange(0, len(value)+1, 1), iterations, linestyle='solid', marker='o')
            
            plt.title("GMRES iterations at each NR iterations \nP2G = {}, GFG = {}, {}  ".format(key[0], key[1], titles[key[2]]), fontsize=fontsize+3)
            plt.xlabel("NR Iterations", fontsize=fontsize)
            plt.ylabel("GMRES Iterations", fontsize=fontsize)
                        
            plt.xlim(-1, len(iterations))
            plt.ylim(-1, max(iterations))
            
            x_step = max(1, 10**int(np.log10(plt.xlim()[1])-1)) * (5 if int(str(int(plt.xlim()[1]))[:2]) > 20 else 1)
            plt.xticks(np.arange(plt.xlim()[0]+1, plt.xlim()[1], x_step, dtype=int))
            y_step = max(1, 10**int(np.log10(plt.ylim()[1])-1)) * (5 if int(str(int(plt.ylim()[1]))[:2]) > 20 else 1)
            plt.ylim(-1, max(iterations)+y_step)
            plt.yticks(np.arange(plt.ylim()[0]+1, plt.ylim()[1], y_step, dtype=int))

            plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
            plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)

            plt.grid()
            
            name = "iterations_clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}_p2g_{}_gfg_{}".format(number_of_clones_e_list[key[2]],
                                                                                                                 number_of_clones_g_list[key[2]],
                                                                                                                 number_of_clones_c, 
                                                                                                                 number_of_merges_e_list[key[2]],
                                                                                                                 number_of_merges_g_list[key[2]],
                                                                                                                 key[0], 
                                                                                                                 key[1])
            os.makedirs(directory, exist_ok=True)
            
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
            

def plot_iterations_krylov_comparison_ilu(network_names_e, network_names_g,
                                          slack_positions_g, slack_position_e,
                                          number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                          p2g_units_max, gfg_units_max,
                                          lin_solver, lin_solver_parameters,
                                          disk, results,
                                          fill_factor_min, fill_factor_max):
    if lin_solver in {'direct', 'lu'}:
        print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
        return None
        
    directory = os.path.join(disk, 
                             results,
                             '{}_{}'.format(network_names_e[0], network_names_g[0]),
                             'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                             lin_solver,
                             'ilu',
                             'iterations_krylov')
        
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    fontsize = 12

    data_iterations = {}

    for fill_factor in range(fill_factor_min, fill_factor_max+1):
        lin_solver_parameters['fill_factor'] = fill_factor
        
        data = collect_residuals_krylov(network_names_e, network_names_g,
                                        slack_positions_g, slack_position_e,
                                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                        p2g_units_max, gfg_units_max,
                                        lin_solver, lin_solver_parameters,
                                        disk, results)
              
        for key, value in data.items():
            if (len(value) > 0):
                data_iterations[(key[0], key[1], key[2], fill_factor)] = [0]
                for row in value:
                    data_iterations[(key[0], key[1], key[2], fill_factor)].append(len(row[:, -1])-1)
        
    for size in set(key[-2] for key in data_iterations.keys()):
        for p2g_units in set(key[0] for key in data_iterations.keys()):
            for gfg_units in set(key[1] for key in data_iterations.keys()):
                plt.figure(figsize=(8, 8))
        
                for fill_factor in set(key[-1] for key in data_iterations.keys()):
                    try:         
                        plt.step(np.arange(0, len(data_iterations[(p2g_units, gfg_units, size, fill_factor)]), 1), 
                                 data_iterations[(p2g_units, gfg_units, size, fill_factor)], 
                                 linestyle='solid', label='k = {}'.format(fill_factor))
                    except:
                        pass
                    
                plt.title("P2G = {}, GFG = {}, {}: GMRES iterations at each NR iterations".format(p2g_units, gfg_units, titles[size]), fontsize=fontsize)
                plt.xlabel("NR Iterations", fontsize=fontsize)
                plt.ylabel("GMRES Iterations", fontsize=fontsize)
    
                plt.xlim(-1, plt.xlim()[1])
                plt.ylim(-1, plt.ylim()[1])
                
                x_step = max(1, 10**int(np.log10(plt.xlim()[1])-1)) * (5 if int(str(int(plt.xlim()[1]))[:2]) > 20 else 1)
                plt.xticks(np.arange(plt.xlim()[0]+1, plt.xlim()[1], x_step, dtype=int))
                y_step = max(1, 10**int(np.log10(plt.ylim()[1])-1)) * (5 if int(str(int(plt.ylim()[1]))[:2]) > 20 else 1)
                plt.ylim(-1, plt.ylim()[1]+y_step)
                plt.yticks(np.arange(plt.ylim()[0]+1, plt.ylim()[1], y_step, dtype=int))

                plt.grid()
                
                plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=fontsize-2)
        
                name = "comparison_ilu_clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}_p2g_{}_gfg_{}".format(number_of_clones_e_list[size],
                                                                                                                         number_of_clones_g_list[size],
                                                                                                                         number_of_clones_c, 
                                                                                                                         number_of_merges_e_list[size],
                                                                                                                         number_of_merges_g_list[size],
                                                                                                                         p2g_units, 
                                                                                                                         gfg_units)
                os.makedirs(directory, exist_ok=True)
                
                plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
                plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
        
    return None


def plot_total_iterations_krylov_comparison_ilu(network_names_e, network_names_g,
                                                slack_positions_g, slack_position_e,
                                                number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                p2g_units_max, gfg_units_max,
                                                lin_solver, lin_solver_parameters,
                                                disk, results,
                                                fill_factor_min, fill_factor_max):
    if lin_solver in {'direct', 'lu'}:
        print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
        return None
        
    directory = os.path.join(disk, 
                             results,
                             '{}_{}'.format(network_names_e[0], network_names_g[0]),
                             'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                             lin_solver,
                             'ilu',
                             'iterations_krylov')
        
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    fontsize = 12

    data_iterations = {}
    data_unknowns_dict = {}

    for fill_factor in range(fill_factor_min, fill_factor_max+1):
        lin_solver_parameters['fill_factor'] = fill_factor

        _, data_unknowns = collect_residuals(network_names_e, network_names_g,
                                             slack_positions_g, slack_position_e,
                                             number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                             p2g_units_max, gfg_units_max,
                                             lin_solver, lin_solver_parameters,
                                             disk, results)
        
        data = collect_residuals_krylov(network_names_e, network_names_g,
                                        slack_positions_g, slack_position_e,
                                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                        p2g_units_max, gfg_units_max,
                                        lin_solver, lin_solver_parameters,
                                        disk, results)
              
        for key, value in data.items():
            try:
                data_unknowns_dict[(key[0], key[1], key[2], fill_factor)] = data_unknowns[(key[0], key[1], key[2])]
            except:
                data_unknowns_dict[(key[0], key[1], key[2], fill_factor)] = None
            if (len(value) > 0):
                data_iterations[(key[0], key[1], key[2], fill_factor)] = [0]
                for row in value:
                    data_iterations[(key[0], key[1], key[2], fill_factor)].append(len(row[:, -1])-1)
                
    
    total_iterations = {}

    for p2g_units in set(key[0] for key in data_iterations.keys()):
        for gfg_units in set(key[1] for key in data_iterations.keys()):
            plt.figure(figsize=(8, 8))
            
            for fill_factor in set(key[-1] for key in data_iterations.keys()):
                total_iterations[(p2g_units, gfg_units, fill_factor)] = []
                
                for size in set(key[-2] for key in data_iterations.keys()):
                    try:
                        total_iterations[(p2g_units, gfg_units, fill_factor)].append(sum(data_iterations[(p2g_units, gfg_units, size, fill_factor)]))
                    except:
                        total_iterations[(p2g_units, gfg_units, fill_factor)].append(None)
                
                try:
                    sizes = set(key[-2] for key in data_iterations.keys() if key[0] == p2g_units and key[1] == gfg_units)
                    plt.plot([data_unknowns_dict[(p2g_units, gfg_units, size, fill_factor)] for size in sizes], 
                             total_iterations[(p2g_units, gfg_units, fill_factor)], 
                             linestyle='solid', label='k = {}'.format(fill_factor))
                except:
                    pass
                    
            plt.title("P2G = {}, GFG = {}: GMRES iterations per network size".format(p2g_units, gfg_units), fontsize=fontsize)
            plt.xlabel("Network Size", fontsize=fontsize)
            plt.ylabel("GMRES Iterations", fontsize=fontsize)

            plt.xscale('log', base=10)
            plt.xlim((10**int(np.log10(plt.xlim()[0]-1)), 10**int(np.log10(plt.xlim()[1])+1)))
            
            plt.yscale('log', base=10)
            plt.ylim((1, 10**int(np.log10(plt.ylim()[1])+1)))

            plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
            plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)

            plt.grid()
            
            plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=fontsize-2)
    
            name = "comparison_ilu_total_iterations_p2g_{}_gfg_{}".format(p2g_units, 
                                                                          gfg_units)
            
            os.makedirs(directory, exist_ok=True)
            
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
        
    return None
        

def plot_total_iterations_krylov_comparison_size(network_names_e, network_names_g,
                                                 slack_positions_g, slack_position_e,
                                                 number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                 p2g_units_max, gfg_units_max,
                                                 lin_solver, lin_solver_parameters,
                                                 disk, results):
    
    if lin_solver in {'direct', 'lu'}:
        print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
        return None
    else:
        try:
            if lin_solver_parameters['preconditioner'] == 'ilu':
                preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
            else:
                preconditioner_tag = lin_solver_parameters['preconditioner']
        except:
            pass
        
        directory = os.path.join(disk, 
                                 results,
                                 '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                 'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                 lin_solver,
                                 preconditioner_tag,
                                 'iterations_krylov')
        
    data_residuals, data_unknowns = collect_residuals(network_names_e, network_names_g,
                                                      slack_positions_g, slack_position_e,
                                                      number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                      p2g_units_max, gfg_units_max,
                                                      lin_solver, lin_solver_parameters,
                                                      disk, results)

    data = collect_residuals_krylov(network_names_e, network_names_g,
                                    slack_positions_g, slack_position_e,
                                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                    p2g_units_max, gfg_units_max,
                                    lin_solver, lin_solver_parameters,
                                    disk, results)
     
    titles = ["({}, {}) ({}, {})".format(clones_e, merges_e, clones_g, merges_g) for clones_e, merges_e, clones_g, merges_g in zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)]
    fontsize = 12

    total_gmres_iterations = {}
    unknowns_min = [value for value in data_unknowns.values()][0]
    unknowns_max = 0
    
    # general
    plt.figure(figsize=(6, 6))
    
    cmap = mpl.colormaps['turbo']
    colors = cmap(np.linspace(0, 1, (p2g_units_max + 1)*(gfg_units_max + 1)))
    
    i = 0
    for p2g_units in set(key[0] for key in data.keys()):
        for gfg_units in set(key[1] for key in data.keys()):
            for size in set(key[-2] for key in data.keys()):
                if len(data.get((p2g_units, gfg_units, size))) > 0:
                    total_gmres_iterations[(p2g_units, gfg_units, size)] = 0
                    for data_per_nr_iteration in data[(p2g_units, gfg_units, size)]:
                        total_gmres_iterations[(p2g_units, gfg_units, size)] += data_per_nr_iteration.shape[0]-1
                    
                    if unknowns_min > data_unknowns[(p2g_units, gfg_units, size)]:
                        unknowns_min = data_unknowns[(p2g_units, gfg_units, size)]
                    if unknowns_max < data_unknowns[(p2g_units, gfg_units, size)]:
                        unknowns_max = data_unknowns[(p2g_units, gfg_units, size)]
            
            sizes = set(key[-1] for key in total_gmres_iterations.keys() if key[0] == p2g_units and key[1] == gfg_units)
            unknowns = [data_unknowns[(p2g_units, gfg_units, size)] for size in sizes if data_residuals[(p2g_units, gfg_units, max(sizes))][-1] < 10**-4]
            iterations = [total_gmres_iterations[(p2g_units, gfg_units, size)] for size in sizes if data_residuals[(p2g_units, gfg_units, max(sizes))][-1] < 10**-4]

            if len(unknowns) > 0:
                plt.plot(unknowns, iterations, 
                        linestyle='solid', marker='.', # color=colors[i],
                        label='P2G = {} | GFG = {}'.format(p2g_units, gfg_units))
            
            i += 1

    plt.plot(10**np.arange(int(np.log10(unknowns_min)), int(np.log10(unknowns_max)+2)), 
             10**(2*np.arange(int(np.log10(unknowns_min)), int(np.log10(unknowns_max)+2))-7), 
             linestyle='--', label=r'$O(n^2)$')
        
    plt.title("Total number of GMRES iterations over all NR iterations", fontsize=fontsize)
    plt.xlabel("Network size", fontsize=fontsize)
    plt.ylabel("GMRES Iterations", fontsize=fontsize)

    plt.xscale('log', base=10)
    plt.xlim((10**int(np.log10(unknowns_min)), 10**int(np.log10(unknowns_max)+1)))
    
    plt.yscale('log', base=10)
    plt.ylim((1, 10**int(np.log10(plt.ylim()[1])+1)))

    plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
    plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)

    plt.grid()
    
    plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=fontsize-2)
            
    name = "total_iterations_clones_c_{}".format(number_of_clones_c)
    
    os.makedirs(directory, exist_ok=True)
    
    plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
    plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
    
    
    # order of magnitude difference of d <= 1 in total GMRES iteration with smallest network and network with largest number of iterations
    plt.figure(figsize=(6, 6))
    
    for p2g_units in set(key[0] for key in data.keys()):
        for gfg_units in set(key[1] for key in data.keys()):
            sizes = set(key[-1] for key in total_gmres_iterations.keys() if key[0] == p2g_units and key[1] == gfg_units)
            total_gmres_iterations_sizes = [total_gmres_iterations[(p2g_units, gfg_units, size)] for size in sizes if data_residuals[(p2g_units, gfg_units, size)][-1] < 10**-4]
            try:
                if max(total_gmres_iterations_sizes) / total_gmres_iterations_sizes[0] < 10:
                    plt.plot([data_unknowns[(p2g_units, gfg_units, size)] for size in sizes if data_residuals[(p2g_units, gfg_units, size)][-1] < 10**-4], total_gmres_iterations_sizes, 
                            linestyle='solid', marker='.',
                            label='P2G = {} | GFG = {}'.format(p2g_units, gfg_units))
            except:
                pass
                
    plt.plot(10**np.arange(int(np.log10(unknowns_min)), int(np.log10(unknowns_max)+2)), 
             10**(np.arange(int(np.log10(unknowns_min)), int(np.log10(unknowns_max)+2))-3), 
             linestyle='--', label=r'$O(n^1)$')
    
    plt.xscale('log', base=10)
    plt.xlim((10**int(np.log10(unknowns_min)), 10**int(np.log10(unknowns_max)+1)))
    
    plt.yscale('log', base=10)
    plt.ylim((1, 10**int(np.log10(plt.ylim()[1])+1)))

    plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
    plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)

    plt.grid()
    
    plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=fontsize-2)

    plt.title(r"Total number of GMRES iterations with order of magnitude $d \leq 1$", fontsize=fontsize)
    plt.xlabel("Network size", fontsize=fontsize)
    plt.ylabel("GMRES Iterations", fontsize=fontsize)
    
    name = "total_iterations_clones_c_{}_d_smaller_than_1".format(number_of_clones_c)
    
    plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
    plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
    

    # order of magnitude difference of 1 <= d < 2 in total GMRES iteration with smallest network and network with largest number of iterations
    plt.figure(figsize=(6, 6))
    
    for p2g_units in set(key[0] for key in data.keys()):
        for gfg_units in set(key[1] for key in data.keys()):
            sizes = set(key[-1] for key in total_gmres_iterations.keys() if key[0] == p2g_units and key[1] == gfg_units)
            total_gmres_iterations_sizes = [total_gmres_iterations[(p2g_units, gfg_units, size)] for size in sizes if data_residuals[(p2g_units, gfg_units, size)][-1] < 10**-4]
            try:
                if 10 <= max(total_gmres_iterations_sizes) / total_gmres_iterations_sizes[0] < 10**2:
                    plt.plot([data_unknowns[(p2g_units, gfg_units, size)] for size in sizes if data_residuals[(p2g_units, gfg_units, size)][-1] < 10**-4], total_gmres_iterations_sizes, 
                            linestyle='solid', marker='.',
                            label='P2G = {} | GFG = {}'.format(p2g_units, gfg_units))
            except:
                pass
                
    
    plt.plot(10**np.arange(int(np.log10(unknowns_min)), int(np.log10(unknowns_max)+2)), 
             10**(1.5*np.arange(int(np.log10(unknowns_min)), int(np.log10(unknowns_max)+2))-5), 
             linestyle='--', label=r'$O(n^{1.5})$')

    plt.xscale('log', base=10)
    plt.xlim((10**int(np.log10(unknowns_min)), 10**int(np.log10(unknowns_max)+1)))

    plt.yscale('log', base=10)
    plt.ylim((1, 10**int(np.log10(plt.ylim()[1])+1)))

    plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
    plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)

    plt.grid()
    
    plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=fontsize-2)

    plt.title(r"Total number of GMRES iterations with order of magnitude difference $1 \leq d < 2$", fontsize=fontsize)
    plt.xlabel("Network size", fontsize=fontsize)
    plt.ylabel("GMRES Iterations", fontsize=fontsize)
    
    name = "total_iterations_clones_c_{}_d_in_between_1_and_2".format(number_of_clones_c)
    
    plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
    plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
    
    
    # order of magnitude difference of d >= 2 in total GMRES iteration with smallest network and network with largest number of iterations
    plt.figure(figsize=(6, 6))
    
    for p2g_units in set(key[0] for key in data.keys()):
        for gfg_units in set(key[1] for key in data.keys()):
            sizes = set(key[-1] for key in total_gmres_iterations.keys() if key[0] == p2g_units and key[1] == gfg_units)
            total_gmres_iterations_sizes = [total_gmres_iterations[(p2g_units, gfg_units, size)] for size in sizes if data_residuals[(p2g_units, gfg_units, size)][-1] < 10**-4]
            try:
                if max(total_gmres_iterations_sizes) / total_gmres_iterations_sizes[0] >= 10**2:
                    plt.plot([data_unknowns[(p2g_units, gfg_units, size)] for size in sizes if data_residuals[(p2g_units, gfg_units, size)][-1] < 10**-4], total_gmres_iterations_sizes, 
                            linestyle='solid', marker='.',
                            label='P2G = {} | GFG = {}'.format(p2g_units, gfg_units))
            except:
                pass
                
    plt.plot(10**np.arange(int(np.log10(unknowns_min)), int(np.log10(unknowns_max)+2)), 
             10**(2*np.arange(int(np.log10(unknowns_min)), int(np.log10(unknowns_max)+2))-7), 
             linestyle='--', label=r'$O(n^2)$')

    plt.xscale('log', base=10)
    plt.xlim((10**int(np.log10(unknowns_min)), 10**int(np.log10(unknowns_max)+1)))
    
    plt.yscale('log', base=10)
    plt.ylim((1, 10**int(np.log10(plt.ylim()[1])+1)))

    plt.tick_params(axis='both', which='major', labelsize=fontsize-2)
    plt.tick_params(axis='both', which='minor', labelsize=fontsize-2)

    plt.grid()
    
    plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=fontsize-2)

    plt.title(r"Total number of GMRES iterations with order of magnitude difference $d \geq 2$", fontsize=fontsize)
    plt.xlabel("Network size", fontsize=fontsize)
    plt.ylabel("GMRES Iterations", fontsize=fontsize)
    
    name = "total_iterations_clones_c_{}_d_larger_equal_than_2".format(number_of_clones_c)
    
    plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
    plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")


def collect_summary_lu(network_names_e, network_names_g,
                       slack_positions_g, slack_position_e,
                       number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                       p2g_units_max, gfg_units_max,
                       lin_solver, lin_solver_parameters,
                       disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass
 
    header = ""
    rows = {}
    rows_difference = {}

    data = {}
    data_unknowns = {}

    for i, (number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g) in enumerate(zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)):
        header += " & ({:d}, {:d}) ({:d}, {:d})".format(number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g)
        for network_name_e in network_names_e:            
            for network_name_g in network_names_g:
                for p2g_units in range(0, p2g_units_max+1):
                    for gfg_units in range(0, gfg_units_max+1):
                        for slack_position_g in slack_positions_g:                    
                            if lin_solver in {'direct', 'lu'}:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver)
                            else:
                                directory = os.path.join(disk, 
                                                         results,
                                                         '{}_{}'.format(network_name_e, network_name_g),
                                                         'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
                                                         'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
                                                                                                                              number_of_clones_g,
                                                                                                                              number_of_clones_c, 
                                                                                                                              number_of_merges_e, 
                                                                                                                              number_of_merges_g),
                                                         'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
                                                         lin_solver,
                                                         preconditioner_tag)
                            
                            if rows.get((p2g_units, gfg_units)) is None:
                                rows[(p2g_units, gfg_units)] = ""
                            if rows_difference.get((p2g_units, gfg_units)) is None:
                                rows_difference[(p2g_units, gfg_units)] = ""
                            
                            try:
                                data[(p2g_units, gfg_units, i)] = np.genfromtxt(os.path.join(directory, 'summary_lu.txt'),  dtype=float, delimiter='\n')
                            except:
                                pass

                            try:
                                summary = np.genfromtxt(os.path.join(directory, 'summary.txt'),  dtype=str, delimiter='\n')
                                residual = float(summary[2].split()[-1])
                                coupling_units_okay = float(summary[-3].split()[-1])
                                unknowns =  float(summary[0].split()[-1])
                                
                                data_unknowns[(p2g_units, gfg_units, i)] = int(unknowns)

                                rows[(p2g_units, gfg_units)] += " & \\cellcolor{{green!50}} {:4.3f}".format(data[(p2g_units, gfg_units, i)][3])
                                                                                                                    
                                if (residual < 1e-6) and (coupling_units_okay == 1):
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{green!50}} {:4.3f}".format(data[(p2g_units, gfg_units, i)][3])
                                elif (residual < 1e-6) and (coupling_units_okay != 1):
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{orange!50}} {:4.3f}".format(data[(p2g_units, gfg_units, i)][3])
                                else:
                                    rows[(p2g_units, gfg_units)] += " & \\cellcolor{{red!50}} {:4.3f}".format(data[(p2g_units, gfg_units, i)][3])
                            except:
                                rows[(p2g_units, gfg_units)] += " & "
                                rows_difference[(p2g_units, gfg_units)] += " & "

    header = "P2G & GFG" + header
    header += " \\\\ \\hline"
    print(header)

    for key, value in rows.items():
        rows[key] += " \\\\ \\hline"
        rows[key] = "{:d} & {:d}".format(key[0], key[1]) + rows[key]
        print(rows[key])
        
    return data, data_unknowns


def plot_summary_lu(network_names_e, network_names_g, 
                    slack_positions_g, slack_position_e,
                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                    p2g_units_max, gfg_units_max,
                    lin_solver, lin_solver_parameters,
                    disk, results):
    try:
        if lin_solver_parameters['preconditioner'] == 'ilu':
            preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
        else:
            preconditioner_tag = lin_solver_parameters['preconditioner']
    except:
        pass

    data_lu, data_unknowns = collect_summary_lu(network_names_e, network_names_g, 
                                                slack_positions_g, slack_position_e,
                                                number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                p2g_units_max, gfg_units_max,
                                                'lu', dict(),
                                                disk, results)

    data, _ = collect_summary_lu(network_names_e, network_names_g, 
                                 slack_positions_g, slack_position_e,
                                 number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                 p2g_units_max, gfg_units_max,
                                 lin_solver, lin_solver_parameters,
                                 disk, results)
        
    sizes = set([key[-1] for key in data.keys()])
    
    fontsize = 11
    
    for p2g_units in range(p2g_units_max+1):
        for gfg_units in range(gfg_units_max+1):
            ratio_lu = []
            nnz_LU_lu = []

            ratio = []
            nnz_J = []
            nnz_LU = []

            unknowns = []

            for size in sizes:
                if data.get((p2g_units, gfg_units, size)) is not None:
                    ratio_lu.append(data_lu[(p2g_units, gfg_units, size)][3])
                    nnz_LU_lu.append(data_lu[(p2g_units, gfg_units, size)][1] + data_lu[(p2g_units, gfg_units, size)][2])

                    ratio.append(data[(p2g_units, gfg_units, size)][3])
                    nnz_J.append(data[(p2g_units, gfg_units, size)][0])
                    nnz_LU.append(data[(p2g_units, gfg_units, size)][1] + data[(p2g_units, gfg_units, size)][2])

                    unknowns.append(data_unknowns[(p2g_units, gfg_units, size)])
                
            fig, ax = plt.subplots(1, 2, figsize=(12, 5))

            # fill-in ratio
            ax[0].plot(unknowns, ratio_lu, marker='.', label="LU")
            ax[0].plot(unknowns, ratio, marker='.', label="{}({})".format(lin_solver_parameters['preconditioner'],
                                                                          lin_solver_parameters['fill_factor']).upper())
            
            ax[0].set_title("Fill-in ratio", fontsize=fontsize)
            ax[0].set_xlabel("Number of unknowns", fontsize=fontsize)
            ax[0].set_ylabel("Ratio", fontsize=fontsize)

            ax[0].set_xlim([0, ax[0].get_xlim()[1]])
            ax[0].set_ylim([0, ax[0].get_ylim()[1]])
            
            # lowest_order = find_order(x0=unknowns, option='lowest') # int(np.floor(np.log10(unknowns[0])))
            # highest_order = find_order(x0=unknowns, option='highest') # int(np.ceil(np.log10(unknowns[-1])))
            # ax[0].set_xscale('log', base=10)
            # ax[0].set_xticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

            ax[0].tick_params(axis='both', which='major', labelsize=fontsize-1)
            ax[0].tick_params(axis='both', which='minor', labelsize=fontsize-1)
            ax[0].ticklabel_format(style='sci', scilimits=(-4, 4), axis='both')

            ax[0].grid()

            ax[0].legend(loc='lower left', fontsize=fontsize)

            # number of nonzeros
            ax[1].plot(unknowns, nnz_LU_lu, marker='.', label="LU")
            ax[1].plot(unknowns, nnz_LU, marker='.', label="{}({})".format(lin_solver_parameters['preconditioner'],
                                                                           lin_solver_parameters['fill_factor']).upper())
            ax[1].plot(unknowns, nnz_J, marker='.', label="Jacobian")
            
            ax[1].set_xlim([0, ax[1].get_xlim()[1]])
            ax[1].set_ylim([0, ax[1].get_ylim()[1]])

            # lowest_order = find_order(x0=unknowns, option='lowest') # int(np.floor(np.log10(unknowns[0])))
            # highest_order = find_order(x0=unknowns, option='highest') # int(np.ceil(np.log10(unknowns[-1])))
            # ax[1].set_xscale('log', base=10)
            # ax[1].set_xticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

            # lowest_order = find_order(x0=nnz_J, option='lowest') # int(np.floor(np.log10(values[0])))
            # highest_order = find_order(x0=nnz_LU_lu, option='highest') # int(np.ceil(np.log10(values[-1])))
            # ax[1].set_yscale('log', base=10)
            # ax[1].set_yticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{{{}}}$".format(k) for k in range(lowest_order, highest_order+1)])

            ax[1].set_title("Number of nonzeros of factorisation", fontsize=fontsize)
            ax[1].set_xlabel("Number of unknowns", fontsize=fontsize)
            ax[1].set_ylabel("Number of nonzeros", fontsize=fontsize)
            
            ax[1].tick_params(axis='both', which='major', labelsize=fontsize-1)
            ax[1].tick_params(axis='both', which='minor', labelsize=fontsize-1)
            ax[1].ticklabel_format(style='sci', scilimits=(-4, 4), axis='both')

            ax[1].grid()

            ax[1].legend(loc='upper left', fontsize=fontsize)

            plt.suptitle("P2G = {}, GFG = {}".format(p2g_units, gfg_units), fontsize=fontsize+3)
            
            directory = os.path.join(disk, 
                                     results,
                                     '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                     'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                     'lu vs gmres+{}'.format(preconditioner_tag))
                            
            directory = os.path.join(directory, 'fillin_ratio')

            os.makedirs(directory, exist_ok=True)

            name = "fillin_ratio_clones_c_{}_p2g_{}_gfg_{}".format(number_of_clones_c, p2g_units, gfg_units)
            plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
            plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")


def plot_summary_lu_comparison(network_names_e, network_names_g, 
                               slack_positions_g, slack_position_e,
                               number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                               p2g_units_max, gfg_units_max,
                               lin_solver, lin_solver_parameters,
                               disk, results,
                               fill_factor_min, fill_factor_max):

    data_lu, data_unknowns = collect_summary_lu(network_names_e, network_names_g, 
                                                slack_positions_g, slack_position_e,
                                                number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                p2g_units_max, gfg_units_max,
                                                'lu', dict(),
                                                disk, results)

    data_dict = {}

    for fill_factor in range(fill_factor_min, fill_factor_max+1):
        lin_solver_parameters['fill_factor'] = fill_factor

        data, _ = collect_summary_lu(network_names_e, network_names_g, 
                                     slack_positions_g, slack_position_e,
                                     number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                     p2g_units_max, gfg_units_max,
                                     lin_solver, lin_solver_parameters,
                                     disk, results)
                      
        for key, value in data.items():
            data_dict[(key[0], key[1], key[2], fill_factor)] = data
                        
    sizes = set([key[-1] for key in data.keys()])
    
    fontsize = 11
    
    for p2g_units in range(p2g_units_max+1):
        for gfg_units in range(gfg_units_max+1):
            try:
                ratio_lu = []
                nnz_LU_lu = []
                nnz_J = []
                unknowns = []

                ratio = {}
                nnz_LU = {}

                # LU
                for size in sizes:
                    if data.get((p2g_units, gfg_units, size)) is not None:
                        ratio_lu.append(data_lu[(p2g_units, gfg_units, size)][3])
                        nnz_LU_lu.append(data_lu[(p2g_units, gfg_units, size)][1] + data_lu[(p2g_units, gfg_units, size)][2])
                        nnz_J.append(data_lu[(p2g_units, gfg_units, size)][0])
                        unknowns.append(data_unknowns[(p2g_units, gfg_units, size)])

                fig, ax = plt.subplots(1, 2, figsize=(12, 5))

                # fill-in ratio
                ax[0].plot(unknowns, ratio_lu, marker='.', label="LU")
                # number of nonzeros
                ax[1].plot(unknowns, nnz_LU_lu, marker='.', label="LU")
        
                # ILU
                for fill_factor in set(key[-1] for key in data_dict.keys()):
                    ratio[(p2g_units, gfg_units, fill_factor)] = []
                    nnz_LU[(p2g_units, gfg_units, fill_factor)] = []

                    for i, size in enumerate(sizes):
                        if data.get((p2g_units, gfg_units, size)) is not None:
                            ratio[(p2g_units, gfg_units, fill_factor)].append(data[(p2g_units, gfg_units, size)][3])
                            nnz_LU[(p2g_units, gfg_units, fill_factor)].append(data[(p2g_units, gfg_units, size)][1] + data[(p2g_units, gfg_units, size)][2])

                    ax[0].plot(unknowns, ratio[(p2g_units, gfg_units, fill_factor)], 
                            marker='.', label="{}({})".format(lin_solver_parameters['preconditioner'], fill_factor).upper())
                                    
                    ax[1].plot(unknowns, nnz_LU[(p2g_units, gfg_units, fill_factor)], 
                            marker='.', label="{}({})".format(lin_solver_parameters['preconditioner'], fill_factor).upper())
                
                ax[0].set_title("Fill-in ratio", fontsize=fontsize)
                ax[0].set_xlabel("Number of unknowns", fontsize=fontsize)
                ax[0].set_ylabel("Ratio", fontsize=fontsize)

                ax[0].set_xlim([0, ax[0].get_xlim()[1]])
                ax[0].set_ylim([0, ax[0].get_ylim()[1]])
                
                # lowest_order = find_order(x0=unknowns, option='lowest') # int(np.floor(np.log10(unknowns[0])))
                # highest_order = find_order(x0=unknowns, option='highest') # int(np.ceil(np.log10(unknowns[-1])))
                # ax[0].set_xscale('log', base=10)
                # ax[0].set_xticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

                ax[0].tick_params(axis='both', which='major', labelsize=fontsize-1)
                ax[0].tick_params(axis='both', which='minor', labelsize=fontsize-1)
                ax[0].ticklabel_format(style='sci', scilimits=(-4, 4), axis='both')

                ax[0].grid()

                ax[0].legend(loc='lower left', fontsize=fontsize)

                # number of nonzeros
                ax[1].plot(unknowns, nnz_J, marker='.', label="Jacobian")
                
                ax[1].set_xlim([0, ax[1].get_xlim()[1]])
                ax[1].set_ylim([0, ax[1].get_ylim()[1]])

                # lowest_order = find_order(x0=unknowns, option='lowest') # int(np.floor(np.log10(unknowns[0])))
                # highest_order = find_order(x0=unknowns, option='highest') # int(np.ceil(np.log10(unknowns[-1])))
                # ax[1].set_xscale('log', base=10)
                # ax[1].set_xticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

                # lowest_order = find_order(x0=nnz_J, option='lowest') # int(np.floor(np.log10(values[0])))
                # highest_order = find_order(x0=nnz_LU_lu, option='highest') # int(np.ceil(np.log10(values[-1])))
                # ax[1].set_yscale('log', base=10)
                # ax[1].set_yticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{{{}}}$".format(k) for k in range(lowest_order, highest_order+1)])

                ax[1].set_title("Number of nonzeros of factorisation", fontsize=fontsize)
                ax[1].set_xlabel("Number of unknowns", fontsize=fontsize)
                ax[1].set_ylabel("Number of nonzeros", fontsize=fontsize)
                
                ax[1].tick_params(axis='both', which='major', labelsize=fontsize-1)
                ax[1].tick_params(axis='both', which='minor', labelsize=fontsize-1)
                ax[1].ticklabel_format(style='sci', scilimits=(-4, 4), axis='both')

                ax[1].grid()

                ax[1].legend(loc='upper left', fontsize=fontsize)

                plt.suptitle("P2G = {}, GFG = {}".format(p2g_units, gfg_units), fontsize=fontsize+3)
                
                directory = os.path.join(disk, 
                                        results,
                                        '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                        'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                        'lu vs gmres+ilu')
                                
                directory = os.path.join(directory, 'ratio')

                os.makedirs(directory, exist_ok=True)

                name = "fillin_ratio_clones_c_{}_p2g_{}_gfg_{}".format(number_of_clones_c, p2g_units, gfg_units)
                plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
                plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")
            except:
                pass

In [ ]:
# def clean_residuals_krylov(network_names_e, network_names_g,
#                            slack_positions_g, slack_position_e,
#                            number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
#                            p2g_units_max, gfg_units_max,
#                            lin_solver, lin_solver_parameters,
#                            disk, results):
#     try:
#         if lin_solver_parameters['preconditioner'] == 'ilu':
#             preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
#         else:
#             preconditioner_tag = lin_solver_parameters['preconditioner']
#     except:
#         pass
 
#     for i, (number_of_clones_e, number_of_merges_e, number_of_clones_g, number_of_merges_g) in enumerate(zip(number_of_clones_e_list, number_of_merges_e_list, number_of_clones_g_list, number_of_merges_g_list)):
#         for network_name_e in network_names_e:            
#             for network_name_g in network_names_g:
#                 for p2g_units in range(0, p2g_units_max+1):
#                     for gfg_units in range(0, gfg_units_max+1):
#                         for slack_position_g in slack_positions_g:                  
#                             if lin_solver in {'direct', 'lu'}:
#                                 print("lin_solver is not a Krylov solver, but {}".format(lin_solver))
#                                 return None
#                             else:
#                                 data = None
#                                 directory = os.path.join(disk, 
#                                                          results,
#                                                          '{}_{}'.format(network_name_e, network_name_g),
#                                                          'slack_position_e_{}_g_{}'.format(slack_position_e, slack_position_g),
#                                                          'clones_e_{}_clones_g_{}_clones_c_{}_merges_e_{}_merges_g_{}'.format(number_of_clones_e, 
#                                                                                                                               number_of_clones_g,
#                                                                                                                               number_of_clones_c, 
#                                                                                                                               number_of_merges_e, 
#                                                                                                                               number_of_merges_g),
#                                                          'p2g_{}_gfg_{}'.format(p2g_units, gfg_units),
#                                                          lin_solver,
#                                                          preconditioner_tag,
#                                                          'residuals_krylov')
                                
#                                 try:
#                                     for iteration in os.listdir(directory):
#                                         print(os.path.join(directory, '{}'.format(iteration)))
                                        
#                                         try:
#                                             data = np.loadtxt(os.path.join(directory, '{}'.format(iteration)))
#                                             max_outer_iteration = 0
                                            
#                                             for row in range(data.shape[0]):
#                                                 krylov_outer_iteration = data[row][0]
#                                                 if max_outer_iteration <= krylov_outer_iteration:
#                                                     max_outer_iteration = krylov_outer_iteration
#                                                 else:
#                                                     data = data[:row, :]
#                                                     break
                                            
#                                             np.savetxt(os.path.join(directory, '{}'.format(iteration)), data, fmt=['%5d', '%5d', '%24.16e', '%24.16e'])
#                                         except:
#                                             pass
                                        
#                                 except:
#                                     pass                                                                                 

#     return None

# network_names_e = ['case9241pegase']
# network_names_g = ['GasLib-4197']

# slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
# slack_positions_g = [11]

# number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
# number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

# number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
# number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

# number_of_clones_c = 8

# p2g_units_max = 3
# gfg_units_max = 4

# lin_solver = 'gmres'

# lin_solver_parameters = {}
# lin_solver_parameters['preconditioner'] = 'ilu'
# lin_solver_parameters['fill_factor'] = 5

# disk = r'E:'
# results = 'results'

# clean_residuals_krylov(network_names_e, network_names_g, 
#                        slack_positions_g, slack_position_e,
#                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
#                        p2g_units_max, gfg_units_max,
#                        lin_solver, lin_solver_parameters,
#                        disk, results)

# Relative difference of coupling unit values

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

# number_of_clones_c_list = [1, 2, 4, 8, 16, 32]
number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'lu'

lin_solver_parameters = {}

disk = r'E:'
results = 'results'

collect_difference(network_names_e, network_names_g, 
                   slack_positions_g, slack_position_e,
                   number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list,
                   p2g_units_max, gfg_units_max,
                   lin_solver, lin_solver_parameters,
                   disk, results)

# Overview of convergence LU

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

# number_of_clones_c_list = [1, 2, 4, 8, 16, 32]
number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'lu'

lin_solver_parameters = {}

disk = r'E:'
results = 'results'

collect_convergence(network_names_e, network_names_g, 
                    slack_positions_g, slack_position_e,
                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                    p2g_units_max, gfg_units_max,
                    lin_solver, lin_solver_parameters,
                    disk, results)

# Overview of convergence LU with fixed size

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c_list = [1, 2, 4, 8, 16, 32]

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'lu'

lin_solver_parameters = {}

disk = r'E:'
results = 'results'

collect_convergence_fixed_size(network_names_e, network_names_g, 
                               slack_positions_g, slack_position_e,
                               number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c_list,
                               p2g_units_max, gfg_units_max,
                               lin_solver, lin_solver_parameters,
                               disk, results)

# Overview of convergence GMRES+ILU(5)

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32][:-1]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64][:-1]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32][:-1]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4][:-1]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5

disk = r'E:'
results = 'results'

collect_convergence(network_names_e, network_names_g, 
                    slack_positions_g, slack_position_e,
                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                    p2g_units_max, gfg_units_max,
                    lin_solver, lin_solver_parameters,
                    disk, results)

# Overview of convergence with fixed size GMRES+ILU(5)

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32][:-1]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64][:-1]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32][:-1]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4][:-1]

number_of_clones_c_list = [1, 2, 4, 8, 16, 32][:-1]

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5

disk = r'E:'
results = 'results'

collect_convergence_fixed_size(network_names_e, network_names_g, 
                               slack_positions_g, slack_position_e,
                               number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c_list,
                               p2g_units_max, gfg_units_max,
                               lin_solver, lin_solver_parameters,
                               disk, results)

# Computational time LU

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'lu'

lin_solver_parameters = {}
    
disk = r'E:'
results = 'results'

plot_computational_time(network_names_e, network_names_g, 
                        slack_positions_g, slack_position_e,
                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                        p2g_units_max, gfg_units_max,
                        lin_solver, lin_solver_parameters,
                        disk, results)

# Computational time GMRES+ILU(5)

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16]
number_of_merges_e_list = [0, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16]
number_of_merges_g_list = [0, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5

disk = r'E:'
results = 'results'

plot_computational_time(network_names_e, network_names_g, 
                        slack_positions_g, slack_position_e,
                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                        p2g_units_max, gfg_units_max,
                        lin_solver, lin_solver_parameters,
                        disk, results)

# Computational time LU vs. GMRES+ILU(5)

In [ ]:
# Model parameters
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

disk = r'E:'
results = 'results'


# LU
lin_solver_lu = 'lu'
lin_solver_parameters = {}
    
data_lu, data_average_lu, data_unknowns_lu = collect_computational_time(network_names_e, network_names_g,
                                                                        slack_positions_g, slack_position_e,
                                                                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                                        p2g_units_max, gfg_units_max,
                                                                        lin_solver_lu, lin_solver_parameters,
                                                                        disk, results)


# GMRES + ILU(5)
lin_solver_gmres = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5

data_gmres, data_average_gmres, data_unknowns_gmres = collect_computational_time(network_names_e, network_names_g,
                                                                                 slack_positions_g, slack_position_e,
                                                                                 number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                                                 p2g_units_max, gfg_units_max,
                                                                                 lin_solver_gmres, lin_solver_parameters,
                                                                                 disk, results)


# Plotting
try:
    if lin_solver_parameters['preconditioner'] == 'ilu':
        preconditioner_tag = "{}({})".format(lin_solver_parameters['preconditioner'], lin_solver_parameters['fill_factor'])
    else:
        preconditioner_tag = lin_solver_parameters['preconditioner']
except:
    pass

sizes_lu = set([key[-1] for key in data_lu.keys()])
sizes_gmres = set([key[-1] for key in data_gmres.keys()])

fontsize = 12

for p2g_units in range(p2g_units_max+1):
    for gfg_units in range(gfg_units_max+1):        
        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        
        values_lu = []
        values_average_lu = []
        unknowns_lu = []
        for size in sizes_lu:
            if data_lu.get((p2g_units, gfg_units, size)) is not None:
                if data_lu.get((p2g_units, gfg_units, size)) > 0:
                    values_lu.append(data_lu[(p2g_units, gfg_units, size)])
                    values_average_lu.append(data_average_lu[(p2g_units, gfg_units, size)])
                    unknowns_lu.append(data_unknowns_lu[(p2g_units, gfg_units, size)])
            
        values_gmres = []
        values_average_gmres = []
        unknowns_gmres = []
        for size in sizes_gmres:
            if data_gmres.get((p2g_units, gfg_units, size)) is not None:
                if data_gmres.get((p2g_units, gfg_units, size)) > 0:
                    values_gmres.append(data_gmres[(p2g_units, gfg_units, size)])
                    values_average_gmres.append(data_average_gmres[(p2g_units, gfg_units, size)])
                    unknowns_gmres.append(data_unknowns_gmres[(p2g_units, gfg_units, size)])
            
          
        # total computational time
        ax[0].loglog(unknowns_lu, values_lu, marker='.', label="{}".format(lin_solver_lu))
        ax[0].loglog(unknowns_gmres, values_gmres, marker='.', label="{}+{}".format(lin_solver_gmres, preconditioner_tag))

        lowest_order = find_order(x0=unknowns_lu, x1=unknowns_gmres, option='lowest') # int(min(np.floor(np.log10(unknowns_lu[0])), np.floor(np.log10(unknowns_gmres[0]))))
        highest_order = find_order(x0=unknowns_lu, x1=unknowns_gmres, option='highest') # int(max(np.ceil(np.log10(unknowns_lu[-1])),  np.ceil(np.log10(unknowns_gmres[-1]))))
        
        ax[0].loglog([10**k for k in range(lowest_order, highest_order+1)], [10**(k-3.5) for k in range(lowest_order, highest_order+1)], 
                        linestyle='--', label="$O(n)$")
        ax[0].loglog([10**k for k in range(lowest_order, highest_order+1)], [10**(2*k-7.5) for k in range(lowest_order, highest_order+1)], 
                        linestyle='--', label="$O(n^2)$")

        ax[0].set_xticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

        lowest_order = find_order(x0=values_lu, x1=values_gmres, option='lowest') # int(min(np.floor(np.log10(values_lu[0])), np.floor(np.log10(values_gmres[0]))))
        highest_order = find_order(x0=values_lu, x1=values_gmres, option='highest') # int(max(np.ceil(np.log10(values_lu[-1])), np.ceil(np.log10(values_gmres[-1]))))
        
        ax[0].set_yticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

        ax[0].set_title("Total computational time of NR", fontsize=fontsize)
        ax[0].set_xlabel("Number of unknowns", fontsize=fontsize)
        ax[0].set_ylabel("Time in seconds", fontsize=fontsize)

        ax[0].tick_params(axis='both', which='major', labelsize=fontsize)
        ax[0].tick_params(axis='both', which='minor', labelsize=fontsize)

        ax[0].grid()

        ax[0].legend(loc='upper left', fontsize=fontsize)

        # average linear time
        ax[1].loglog(unknowns_lu, values_average_lu, marker='.', label="{}".format(lin_solver_lu))
        ax[1].loglog(unknowns_gmres, values_average_gmres, marker='.', label="{}+{}".format(lin_solver_gmres, preconditioner_tag))

        lowest_order = find_order(x0=unknowns_lu, x1=unknowns_gmres, option='lowest') # int(min(np.floor(np.log10(unknowns_lu[0])), np.floor(np.log10(unknowns_gmres[0]))))
        highest_order = find_order(x0=unknowns_lu, x1=unknowns_gmres, option='highest') # int(max(np.ceil(np.log10(unknowns_lu[-1])),  np.ceil(np.log10(unknowns_gmres[-1]))))                    
                                
        ax[1].loglog([10**k for k in range(lowest_order, highest_order+1)], [10**(k-5) for k in range(lowest_order, highest_order+1)], 
                        linestyle='--', label="$O(n)$")
        ax[1].loglog([10**k for k in range(lowest_order, highest_order+1)], [10**(2*k-9) for k in range(lowest_order, highest_order+1)], 
                        linestyle='--', label="$O(n^2)$")

        ax[1].set_xticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{}$".format(k) for k in range(lowest_order, highest_order+1)])

        lowest_order = find_order(x0=values_average_lu, x1=values_average_gmres, option='lowest') # int(min(np.floor(np.log10(values_average_lu[0])), np.floor(np.log10(values_average_gmres[0]))))
        highest_order = find_order(x0=values_average_lu, x1=values_average_gmres, option='highest') # int(max(np.ceil(np.log10(values_average_lu[-1])), np.ceil(np.log10(values_average_gmres[-1]))))
                                
        ax[1].set_yticks(ticks=[10**k for k in range(lowest_order, highest_order+1)], labels=["$10^{{{}}}$".format(k) for k in range(lowest_order, highest_order+1)])

        ax[1].set_title("Average time of linear solver", fontsize=fontsize)
        ax[1].set_xlabel("Number of unknowns", fontsize=fontsize)
        ax[1].set_ylabel("Time in seconds", fontsize=fontsize)
        
        ax[1].tick_params(axis='both', which='major', labelsize=fontsize)
        ax[1].tick_params(axis='both', which='minor', labelsize=fontsize)

        ax[1].grid()

        ax[1].legend(loc='upper left', fontsize=fontsize)

        plt.suptitle("P2G = {}, GFG = {}".format(p2g_units, gfg_units), fontsize=fontsize+3)
        
        directory = os.path.join(disk, 
                                 results,
                                 '{}_{}'.format(network_names_e[0], network_names_g[0]),
                                 'slack_position_e_{}_g_{}'.format(slack_position_e, slack_positions_g[0]),
                                 '{} vs {}+{}'.format(lin_solver_lu, lin_solver_gmres, preconditioner_tag))
                    
        os.makedirs(directory, exist_ok=True)

        name = "total_computational_time_NR_and_average_linear_time_comparison_clones_c_{}_p2g_{}_gfg_{}".format(number_of_clones_c, p2g_units, gfg_units)
        plt.savefig(os.path.join(directory, "{}.png".format(name)), bbox_inches="tight")
        plt.savefig(os.path.join(directory, "{}.svg".format(name)), bbox_inches="tight")

# Residuals LU with constant size

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'lu'

lin_solver_parameters = {}
    
disk = r'E:'
results = 'results'

plot_residuals_fixed_size(network_names_e, network_names_g,
                          slack_positions_g, slack_position_e,
                          number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                          p2g_units_max, gfg_units_max,
                          lin_solver, lin_solver_parameters,
                          disk, results)

# Residuals LU with constant coupling configuration

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'lu'

lin_solver_parameters = {}
    
disk = r'E:'
results = 'results'

plot_residuals_fixed_coupling(network_names_e, network_names_g,
                              slack_positions_g, slack_position_e,
                              number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                              p2g_units_max, gfg_units_max,
                              lin_solver, lin_solver_parameters,
                              disk, results)

# Residuals GMRES + ILU(5) with constant size

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_residuals_fixed_size(network_names_e, network_names_g,
                          slack_positions_g, slack_position_e,
                          number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                          p2g_units_max, gfg_units_max,
                          lin_solver, lin_solver_parameters,
                          disk, results)

# Residuals GMRES + ILU(5) with constant coupling configuration

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_residuals_fixed_coupling(network_names_e, network_names_g,
                              slack_positions_g, slack_position_e,
                              number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                              p2g_units_max, gfg_units_max,
                              lin_solver, lin_solver_parameters,
                              disk, results)

# GMRES + ILU(5) Krylov preconditioned residuals per coupling case

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_preconditioned_residuals_krylov_per_case(network_names_e, network_names_g,
                                              slack_positions_g, slack_position_e,
                                              number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                              p2g_units_max, gfg_units_max,
                                              lin_solver, lin_solver_parameters,
                                              disk, results)

# GMRES + ILU(5) Krylov preconditioned residuals per coupling case at all NR iterations

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_preconditioned_residuals_krylov_at_all_NR_iterations(network_names_e, network_names_g,
                                                          slack_positions_g, slack_position_e,
                                                          number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                          p2g_units_max, gfg_units_max,
                                                          lin_solver, lin_solver_parameters,
                                                          disk, results)

# GMRES + ILU(5) Krylov residuals per coupling case at all NR iterations

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_absolute_residuals_krylov_per_case(network_names_e, network_names_g,
                                        slack_positions_g, slack_position_e,
                                        number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                        p2g_units_max, gfg_units_max,
                                        lin_solver, lin_solver_parameters,
                                        disk, results)

# GMRES + ILU(5) Krylov residuals per coupling case at all NR iterations

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_absolute_residuals_krylov_at_all_NR_iterations(network_names_e, network_names_g,
                                                    slack_positions_g, slack_position_e,
                                                    number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                                    p2g_units_max, gfg_units_max,
                                                    lin_solver, lin_solver_parameters,
                                                    disk, results)

# GMRES + ILU(5) Krylov iterations per case

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 16

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_iterations_krylov_per_case(network_names_e, network_names_g,
                                slack_positions_g, slack_position_e,
                                number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                p2g_units_max, gfg_units_max,
                                lin_solver, lin_solver_parameters,
                                disk, results)

# Iterations GMRES + ILU(5) per coupling case

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 16

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_total_iterations_krylov_comparison_size(network_names_e, network_names_g,
                                             slack_positions_g, slack_position_e,
                                             number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                             p2g_units_max, gfg_units_max,
                                             lin_solver, lin_solver_parameters,
                                             disk, results)

# Iterations GMRES + ILU(k) comparing different k

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
    
disk = r'E:'
results = 'results'

fill_factor_min = 4
fill_factor_max = 10

plot_iterations_krylov_comparison_ilu(network_names_e, network_names_g,
                                      slack_positions_g, slack_position_e,
                                      number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                      p2g_units_max, gfg_units_max,
                                      lin_solver, lin_solver_parameters,
                                      disk, results,
                                      fill_factor_min, fill_factor_max)

# Total iterations GMRES + ILU(k) comparing different k

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4]

number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'
lin_solver_parameters['fill_factor'] = 5
    
disk = r'E:'
results = 'results'

plot_total_iterations_krylov_comparison_ilu(network_names_e, network_names_g,
                                            slack_positions_g, slack_position_e,
                                            number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                                            p2g_units_max, gfg_units_max,
                                            lin_solver, lin_solver_parameters,
                                            disk, results,
                                            fill_factor_min=4, fill_factor_max=10)

# LU vs. ILU fill-in ratio

In [ ]:
network_names_e = ['case9241pegase']
network_names_g = ['GasLib-4197']

slack_position_e = 0 # currently only used for naming folders, has no influence on other functions
slack_positions_g = [11]

number_of_clones_e_list = [1, 2, 4, 8, 16, 32, 64][:-1]
number_of_merges_e_list = [0, 64, 64, 64, 64, 64, 64][:-1]

number_of_clones_g_list = [1, 2, 4, 8, 16, 32, 64][:-1]
number_of_merges_g_list = [0, 4, 4, 4, 4, 4, 4][:-1]

# number_of_clones_c_list = [1, 2, 4, 8, 16, 32]
number_of_clones_c = 1

p2g_units_max = 3
gfg_units_max = 4

lin_solver = 'gmres'

lin_solver_parameters = {}
lin_solver_parameters['preconditioner'] = 'ilu'

fill_factor_min = 4
fill_factor_max = 10

disk = r'E:'
results = 'results'

plot_summary_lu_comparison(network_names_e, network_names_g, 
                           slack_positions_g, slack_position_e,
                           number_of_clones_e_list, number_of_clones_g_list, number_of_merges_e_list, number_of_merges_g_list, number_of_clones_c,
                           p2g_units_max, gfg_units_max,
                           lin_solver, lin_solver_parameters,
                           disk, results,
                           fill_factor_min, fill_factor_max)